# Image processing in the browser

This notebook installs [Pillow](https://pillow.readthedocs.io/) and
[scikit-image](https://scikit-image.org/) at runtime to perform image
processing — edge detection, filtering, segmentation — entirely in your
browser.

Both are C extension packages: Pillow wraps libjpeg, libpng, libtiff, and
zlib; scikit-image uses Cython-compiled routines for performance. Installing
them demonstrates that conda-express handles complex native dependency chains
in WebAssembly.

*Examples adapted from the
[scikit-image gallery](https://scikit-image.org/docs/stable/auto_examples/)
(BSD 3-Clause license) and
[Pillow documentation](https://pillow.readthedocs.io/) (MIT-CMU license).*

## Install image processing libraries

In [ ]:
%load_ext conda_emscripten

In [ ]:
%conda install pillow scikit-image

## Generate a test image

We'll create a synthetic image with geometric shapes — no external files
needed, since MEMFS doesn't persist across reloads.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter

img = Image.new("RGB", (400, 400), "#1e293b")
draw = ImageDraw.Draw(img)

draw.ellipse([50, 50, 200, 200], fill="#3b82f6", outline="#60a5fa", width=3)
draw.rectangle([220, 60, 360, 190], fill="#ef4444", outline="#f87171", width=3)
draw.polygon([(200, 380), (130, 240), (270, 240)], fill="#22c55e", outline="#4ade80", width=3)
draw.ellipse([250, 250, 380, 380], fill="#f59e0b", outline="#fbbf24", width=3)

for i in range(0, 400, 40):
    draw.line([(i, 0), (i, 400)], fill="#334155", width=1)
    draw.line([(0, i), (400, i)], fill="#334155", width=1)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(img)
ax.set_title("Synthetic test image (Pillow)")
ax.axis("off")
plt.tight_layout()
plt.show()
print(f"Image: {img.size[0]}×{img.size[1]}, mode={img.mode}")

## Edge detection

Apply Canny edge detection from scikit-image — a multi-stage algorithm using
Gaussian smoothing, gradient computation, non-maximum suppression, and
hysteresis thresholding.

In [ ]:
from skimage import color, feature, filters

gray = color.rgb2gray(np.array(img))

edges_1 = feature.canny(gray, sigma=1.0)
edges_3 = feature.canny(gray, sigma=3.0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(gray, cmap="gray")
axes[0].set_title("Grayscale")

axes[1].imshow(edges_1, cmap="gray")
axes[1].set_title("Canny (σ=1.0)")

axes[2].imshow(edges_3, cmap="gray")
axes[2].set_title("Canny (σ=3.0)")

for ax in axes:
    ax.axis("off")

fig.suptitle("Edge detection — scikit-image in WebAssembly", fontsize=13)
plt.tight_layout()
plt.show()

## Image filtering with Pillow

Apply convolution filters from Pillow's `ImageFilter` module — these call into
the C-compiled filter kernels.

In [ ]:
filter_ops = [
    ("Original", img),
    ("Gaussian blur", img.filter(ImageFilter.GaussianBlur(radius=5))),
    ("Emboss", img.filter(ImageFilter.EMBOSS)),
    ("Contour", img.filter(ImageFilter.CONTOUR)),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, filtered) in zip(axes, filter_ops):
    ax.imshow(filtered)
    ax.set_title(name)
    ax.axis("off")

fig.suptitle("Image filters — Pillow in WebAssembly", fontsize=13)
plt.tight_layout()
plt.show()

## Segmentation

Use Otsu's thresholding and connected-component labeling to segment the
image into distinct regions.

In [ ]:
from skimage import measure, morphology

threshold = filters.threshold_otsu(gray)
binary = gray > threshold

labels = measure.label(binary)
regions = measure.regionprops(labels)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))

ax1.imshow(gray, cmap="gray")
ax1.set_title("Grayscale")

ax2.imshow(binary, cmap="gray")
ax2.axhline(y=0, color="r", linewidth=0)  # invisible, for legend
ax2.set_title(f"Otsu threshold ({threshold:.2f})")

ax3.imshow(labels, cmap="nipy_spectral")
ax3.set_title(f"Connected components ({labels.max()} regions)")

for region in regions:
    if region.area > 100:
        y0, x0, y1, x1 = region.bbox
        rect = plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                              fill=False, edgecolor="white", linewidth=1.5)
        ax3.add_patch(rect)

for ax in (ax1, ax2, ax3):
    ax.axis("off")

fig.suptitle("Image segmentation — scikit-image in WebAssembly", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Found {len([r for r in regions if r.area > 100])} significant regions "
      f"(area > 100 pixels)")

## What just happened

You installed two image processing libraries with complex C dependency trees
(libjpeg, libpng, libtiff, zlib, Cython extensions), then performed edge
detection, convolution filtering, and image segmentation — all running as
WebAssembly in your browser tab.

The same code works identically with `conda install pillow scikit-image`
on your laptop.